In [1]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load pretrained sentence transformer

In [2]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4074.67it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# Simulate customer support tickets
tickets = [
    "my payment didn't go through",
    "transaction failed during checkout",
    "I can't login to my account",
    "forgot my password need reset",
    "order hasn't arrived yet",
    "package is delayed in shipping",
    "want to return my purchase",
    "how do I get a refund",
    "app keeps crashing on startup",
    "application won't open on my phone"
]

In [4]:
# Generate embeddings — one vector per sentence
embeddings = model.encode(tickets)

In [5]:
print(f"Number of tickets: {len(tickets)}")
print(f"Embedding shape: {embeddings.shape}")
print(f"Single embedding dimensions: {embeddings[0].shape}")

Number of tickets: 10
Embedding shape: (10, 384)
Single embedding dimensions: (384,)


In [6]:
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
# Search query
query = "my credit card charge was rejected"
query_embedding = model.encode([query])  # shape: (1, 384)


In [10]:
# Compare query against all tickets
similarities = cosine_similarity(query_embedding, embeddings)[0]


In [11]:
# Rank tickets by similarity
ranked = sorted(zip(similarities, tickets), reverse=True)

In [12]:
print(f"Query: '{query}'\n")
print("Most similar tickets:")
for score, ticket in ranked:
    print(f"  {score:.4f} — {ticket}")

Query: 'my credit card charge was rejected'

Most similar tickets:
  0.5056 — my payment didn't go through
  0.3887 — how do I get a refund
  0.3691 — transaction failed during checkout
  0.3169 — want to return my purchase
  0.2344 — forgot my password need reset
  0.2329 — package is delayed in shipping
  0.2230 — I can't login to my account
  0.1492 — application won't open on my phone
  0.1281 — order hasn't arrived yet
  0.0229 — app keeps crashing on startup


# Chroma db

In [14]:
import chromadb

In [15]:
# Initialize ChromaDB — local persistent storage
client = chromadb.Client()

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [16]:
# Create a collection — like a table in a regular database
collection = client.create_collection(
    name="support_tickets",
    metadata={"hnsw:space": "cosine"}  # use cosine similarity for search
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [17]:
# Add all tickets with their embeddings
collection.add(
    documents=tickets,                                    # original text
    embeddings=embeddings.tolist(),                       # vectors
    ids=[f"ticket_{i}" for i in range(len(tickets))],   # unique IDs
    metadatas=[{"category": "support"} for _ in tickets] # optional metadata
)


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


In [18]:
print(f"Collection size: {collection.count()}")

Collection size: 10


In [19]:
# Query the vector database
query = "my credit card charge was rejected"
query_embedding = model.encode([query]).tolist()

In [20]:
results = collection.query(
    query_embeddings=query_embedding,
    n_results=3  # return top 3 most similar tickets
)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [21]:
print(f"Query: '{query}'\n")
print("Top 3 results from ChromaDB:")
for i, (doc, distance) in enumerate(zip(
    results['documents'][0], 
    results['distances'][0]
)):
    print(f"  {i+1}. Score: {1-distance:.4f} — {doc}")

Query: 'my credit card charge was rejected'

Top 3 results from ChromaDB:
  1. Score: 0.5056 — my payment didn't go through
  2. Score: 0.3887 — how do I get a refund
  3. Score: 0.3691 — transaction failed during checkout
